In [2]:
!uv pip install -q nltk gensim pyLDAvis unidecode matplotlib seaborn pandas pyarrow

In [3]:
!uv pip install -q langchain-huggingface==0.0.3

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
from nltk.tokenize import word_tokenize
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import html 
import json
import matplotlib.pyplot  as plt
import nltk
import numpy as np
import os
import pandas as pd
import unidecode
import re
import requests
import seaborn as sns
import string
import unidecode
import warnings
import torch
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/onyxia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/onyxia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
class CachedLemmatizer:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.cache = {}  # Manual cache as a dictionary

    def lemmatize(self, word, pos='n'):
        if word in self.cache:
            return self.cache[word]
        else:
            lemmatized_word = self.lemmatizer.lemmatize(word, pos)
            self.cache[word] = lemmatized_word  # Store in cache
            return lemmatized_word


cached_lemmatizer = CachedLemmatizer()

In [11]:
if (torch.cuda.is_available()):
    DEVICE="cuda"
else:
    DEVICE="cpu"

In [12]:
stop_words = set(stopwords.words('french'))

In [13]:
OUTPUT_DIR="intermediate_data"

# Utils preprocessing

In [14]:
DICTIONNARY =  ['accord','entreprise', 'preambule', 'sommaire',  'code', 'syndical', 'responsable', 'representant', 
                'present', 'ca', 'organisation', 'preambule', 'peut', 'etre', 'contrat','travail', 'ressources','humaines', 'mise',
                'ainsi', 'et', 'ou', 'alors','collaborateur', 'ci', 'apres', 'party', 'signataire', 'tout', 'etat', 'cause', 'societe', 
                'notamment','article','activite', 'cette', 'donc', 'si', 'sous', 'disposition', 'convention', 'collective', 'dans', 'a', 'cadre',
                'signataire', 'partie', 'parties', 'entre', 'doit', 'mme', 'mr', 'madame', 'monsieur'
               ]


In [17]:
def preprocess_text(text, lang="french"):
    # décoage HTML
    text = html.unescape(text)
    text = re.sub(r"(?im)^article\s+\d+(\.\d+)*\s*[\.\-\–]*\s*", "", text)
    # nettoyage de tous les cractères spéciaux
    text = re.sub(r"&[a-z]+;", " ", text)
    text = re.sub(r"&#\d+;", " ", text)
    text = re.sub(r"[<>{}\[\]\|\^\~`\"'=]+", " ", text)
    text = re.sub(r"[–—•«»]+", " ", text)  # Tirets longs, puces, guillemets français
    text = re.sub(r"\.{2,}", " ", text) 

    # tokenisation
    words = word_tokenize(text)

    # lemming
    #stemmer = SnowballStemmer(lang)
                  
    wnl = cached_lemmatizer
   
    words_cleaned = []
    for w in words:
        #w_norm = unidecode.unidecode(w.lower())
        w_norm = w.lower()
        if (
            w_norm not in stop_words
            and w_norm not in string.punctuation
            and not re.search(r"[<>]|--+|__+|xx+|==+", w_norm)
            and len(w_norm) > 2
        ):
            words_cleaned.append(wnl.lemmatize(w_norm))
            #words_cleaned.append(stemmer.stem(w_norm))

    return words_cleaned


In [19]:
text = "Révision-- de l&rsquo;accord : &gt;&gt;' Tous les deux ans, les partenaires sociaux se réunissent. << Suivi de l’accord......."
print(preprocess_text(text))

['révision', 'accord', 'tous', 'deux', 'an', 'partenaires', 'sociaux', 'réunissent', 'suivi', 'accord']


In [20]:
text = """
ARTICLE 3.6. Contingent annuel d’heures supplémentaires

Article 3.6.1. Fixation du contingent annuel d’heures supplémentaires

Les parties au présent accord conviennent de fixer le contingent annuel d’heures supplémentaires à 500 heures au jour de la signature du présent accord.

"""

print(preprocess_text(text))

['contingent', 'annuel', 'heures', 'supplémentaires', 'fixation', 'contingent', 'annuel', 'heures', 'supplémentaires', 'party', 'présent', 'accord', 'conviennent', 'fixer', 'contingent', 'annuel', 'heures', 'supplémentaires', '500', 'heures', 'jour', 'signature', 'présent', 'accord']


In [21]:
def normalize(text):
    return unidecode.unidecode(text.lower().strip())

def split_text_by_sentences(text, flagged_sentences):
    """
    Découpe le texte en segments basés sur les titres du sommaire, après normalisation.
    """
    split_texts = []
    positions = []

    normalized_text = normalize(text)

    # On garde un mapping (titre original, position) pour préserver les titres initiaux
    for sentence in flagged_sentences:
        norm_sentence = normalize(sentence)
        pos = normalized_text.find(norm_sentence)
        if pos != -1:
            # On retrouve la position réelle dans le texte original
            real_pos = text.lower().find(sentence.lower())
            if real_pos != -1:
                positions.append(real_pos)

    # Si aucune position trouvée, retourner le texte complet
    if not positions:
        return [text]

    positions = sorted(set(positions))
    positions.insert(0, 0)
    positions.append(len(text))

    for i in range(len(positions) - 1):
        start = positions[i]
        end = positions[i + 1]
        split_texts.append(text[start:end].strip())

    return split_texts

In [22]:
def split_text_with_titles(text, summary_titles):
    chunks = split_text_by_sentences(text, summary_titles)
    result = {}
    for title in summary_titles:
        for chunk in chunks:
            if normalize(title) in normalize(chunk[:len(title)+30]):
                result[title] = chunk.strip()
                break
    return result


In [23]:
def nettoyer_titres(titres):
    titres = [html.unescape(titre) for titre in titres]

    titres = [
        re.sub(
            r"""(?im)                           # mode insensible à la casse
            ^\s*                                # espaces initiaux
            (                                   # groupe à supprimer :
                (article\s+\d+(?:[\.\-]\d+)*)   #   ex: Article 3, Article 3.1.2
                |
                (titre\s+[ivxlcdm]+)            #   ex: Titre IV
                |
                (\d+(?:[\.\-\–]\d+)*\s+)        #   ex: 2-1 , 4.1 , 3 –
            )
            [\.\-\–\:]*\s*                      # éventuels séparateurs
            """,
            "",
            titre.strip(),
            flags=re.IGNORECASE | re.VERBOSE
        )
        for titre in titres
    ]

    # Nettoyage final (espaces, ponctuations parasites)
    titres = [
        titre.replace("\xa0", " ")
        .replace(" :", ":")
        .strip(" :\t\n")
        for titre in titres
    ]

    return titres

In [25]:
model_kwargs = {'device': DEVICE}  
MODEL_NAME_EMBEDDER="BAAI/bge-small-en-v1.5"  #petit modèle en anglais
#MODEL_NAME_EMBEDDER="BAAI/bge-m3" #gros modèle multilingue

embedder = HuggingFaceEmbeddings(
    model_name=MODEL_NAME_EMBEDDER, 
    model_kwargs=model_kwargs,
    show_progress=False
)


phrases_non_metier = [
    "Révision de l’accord",
    "Dénonciation de l’accord",
    "Interprétation de l’accord",
    "Suivi de l’accord",
    "Durée de l’accord",
    "Formalités de publicité et de dépôt",
    "Publicité et dépôt",
    "Date d'effet et durée",
    "Champ d'application",
    "Clause de revoyure", 
    "Information des représentants du personnel", 
    "Dispositions relatives à l’accord",
    "Champ d’application",
    "Commission de suivi", 
    "Pause déjeuner du personnel", 
    "Modification de l'accord",
    "Adhésion", 
    "Information du Comité Social et Economique", 
    "Annexe", 
    "Dispositions finales", 
    "Salariés concernés"
    
]

# Embeddings des phrases non-métier
ref_embeddings = embedder.embed_documents(phrases_non_metier)


def filtre_par_similarite_vectorise(phrases, seuil=0.85):
    if not phrases:
        return []
    phrases =[html.unescape(phrase) for phrase in phrases]
    phrases = nettoyer_titres(phrases)
    phrase_embeddings = embedder.embed_documents(phrases)  
    sims = cosine_similarity(phrase_embeddings, ref_embeddings)

    # On garde les phrases dont la similarité max avec une phrase non-métier est < seuil
    keep_idx = np.max(sims, axis=1) < seuil
    return [phrase for phrase, keep in zip(phrases, keep_idx) if keep]

    
def filtre_chunks_par_titre(section_dict, phrases_non_metier, seuil=0.85): #seuil arbitraire : en tester plsr
    """
    Ne garde que les chunks dont le titre est peu similaire aux phrases non métier.
    """
    if not section_dict:
        return []

    titres = list(section_dict.keys())
    titres = nettoyer_titres(titres)
    titres =[html.unescape(titre) for titre in titres]
    titres = [re.sub(r"\xa0", " ", titre) for titre in titres]
    chunks = list(section_dict.values())

    # Embeddings des titres de section
    titre_embeddings = embedder.embed_documents(titres)
    ref_embeddings = embedder.embed_documents(phrases_non_metier)

    sims = cosine_similarity(titre_embeddings, ref_embeddings)

    # On garde les chunks dont le titre est peu similaire aux phrases non métier
    keep_idx = np.max(sims, axis=1) < seuil
    return [chunk.strip() for chunk, keep in zip(chunks, keep_idx) if keep]



In [26]:
phase_1 ="Publicité et dépôt"
phase_2='ARTICLE 7 - PUBLICITE ET DEPOT'

phase_2_embeddings=embedder.embed_documents(phase_2)

sims = cosine_similarity(ref_embeddings, phase_2_embeddings)

In [27]:
filtre_par_similarite_vectorise(["Article 8 – Révision de l’accord", 'Article 5: Contingent annuel d’heures supplémentaires', 'Article 1 – Champ d’application et bénéficiaires\xa0:']) 

['Contingent annuel d’heures supplémentaires']

In [28]:
#pas parfait 
filtre_par_similarite_vectorise(["Article 8 – Révision de l’accord", 'Article 5: Contingent annuel d’heures supplémentaires', 'Article 4 – Information du Comité Social et Economique', 'Article 5 - Dispositions relatives à l’accord']) 

['Contingent annuel d’heures supplémentaires']

In [29]:
def get_valid_chunks_filtered(section_dict, skip_titles=["préambule", "annexe"], seuil_sim=0.85):
    skip_titles_norm = [normalize(t) for t in skip_titles]

    # supprimer le préambule et avant 
    titles = list(section_dict.keys())
    preamble_idx = next((i for i, t in enumerate(titles) if "préambule" in normalize(t)), -1)
    if preamble_idx != -1:
        titles = titles[preamble_idx + 1:]

    # garder les titres valides uniquement
    valid_titles = [
        t for t in titles if all(skip_kw not in normalize(t) for skip_kw in skip_titles_norm)
    ]
    candidate_dict = {t: section_dict[t] for t in valid_titles}

    # filtrer par similarité des titres
    return filtre_chunks_par_titre(candidate_dict, phrases_non_metier, seuil=seuil_sim)


In [30]:
## exemple d'utilisaton 

mon_dict ={'Preambule :  ': 'Preambule :  \n\nConformément aux dispositions du code du travail, la Direction a invité le CSE, en l’absence d’organisations syndicales représentatives dans la structure  à participer à une négociation collective.\n\nAux termes des réunions en date des 10/10/2022, 16/11/2022 et 12/12/2022 ayant permis de rapprocher les points de vue de chacun, les parties ont abouti à la conclusion du présent accord.',
 'Article 1 – Champ d’application et bénéficiaires\xa0:  ': 'Article 1 – Champ d’application et bénéficiaires\xa0:  \n\nLe présent accord concerne l’ensemble des établissements de l’HADVR.\n\nIl concerne tous les salariés quel que soit leur contrat (CDD ou CDI), quelle que soit leur durée de travail et quel que soit leur métier.\nPar ailleurs, pour répondre aux aspirations des salariés d’une part et aux contraintes inhérentes d’une HAD, les parties se sont accordées pour poursuivre les négociations tout au long de l’année 2023 en vue de la conclusion éventuelle d’un accord sur l’aménagement du temps de travail au sein de la structure.',
 'Article 2\xa0: Rémunération et temps de travail': 'Article 2\xa0: Rémunération et temps de travail',
 '2-1\xa0: Prime de partage de la valeur': "2-1\xa0: Prime de partage de la valeur\n\nDe nombreux investissements matériels et humains ont été réalisés au cours de l’année 2022 pour répondre aux besoins de la structure. Ces investissements auront pour conséquence un budget 2022 non équilibré. \nLe conseil d’administration de la structure, conscient des efforts des professionnels pour poursuivre la montée en charge du nombre de patients accueillis en HAD a répondu favorablement pour le versement d’une prime de partage de la valeur de 300 euros à l’ensemble du personnel excepté la Direction, dans les conditions énoncées ci-après.\n\nAfin de bénéficier des exonérations de cotisation sociales et de l’impôt sur le revenu, est éligible le personnel qui, à la date de versement de la prime, c’est-à-dire au 28 février 2023 :\nLié par un contrat de travail ou d’apprentissage ;\nTravailleurs handicapés liés par un contrat de soutien et d’aide par le travail à un ESAT\xa0;\nLes intérimaires ;\nAyant une rémunération brute inférieure à 3 SMIC conformément aux dispositions légales au cours des 12 mois précédant la date de versement de la prime. \n\nLe salaire annuel brut s’entend de la rémunération annuelle brute (variable et primes inclus) reconstituée en équivalent temps plein sur la période allant de février 2022 à janvier 2023, soit 12 mois. \nIl convient de préciser que la prime versée est calculée au prorata\xa0de la durée de présence effective et du temps de travail contractuel sur la période précitée. \nPar ailleurs et conformément aux dispositions légales, les absences pour congé de maternité, de paternité et d'accueil de l'enfant ou d'adoption, les absences pour congé parental d'éducation, pour enfant malade et pour congé de présence parentale, ainsi que les absences pour accident du travail et maladie professionnelle, sont assimilées à des périodes de présence effective et ne seront donc pas décomptées dans le calcul du temps de travail effectif. \nLa prime sera versée en seule fois avec la paie du mois de février 2023 et figurera sur le bulletin de salaire du mois de versement.\nLa prime ne se substituera à aucun des éléments de rémunération, ni à des augmentations salariales ou prime prévues par un accord, par contrat de travail ou usages en vigueur.",
 '2-2\xa0: Prime «\xa0bas salaires\xa0»\xa0:': '2-2\xa0: Prime «\xa0bas salaires\xa0»\xa0:\n\nLe 28 juin 2022 le Ministre de la transformation et de la fonction publique a annoncé une hausse du point d’indice pour les trois versants de la fonction publique applicable en une fois dès le 1er juillet 2022. Les partenaires sociaux de la branche se sont réunis afin de transposer dans la CCN51 la revalorisation intervenue dans la fonction publique. A l’issue des différentes réunions de négociation qui se sont tenues, aucune organisation syndicale n’a été signataire des textes mis à la signature. La FEHAP a pris une recommandation patronale réévaluant la valeur du point dans la CCN51 en date du 23 novembre 2022. \nDans le contexte inflationniste des derniers mois, compte tenu de la concurrence accrue avec le secteur public, des tensions en matière de recrutement et de la nécessité de fidélisation des professionnels, il est décidé de mettre en en place, par accord d’entreprise, une mesure ciblée pour les «\xa0bas salaires\xa0», en sus de l’augmentation de la valeur du point CCN51.\n\n\nLe conseil d’administration de la structure, conscient que la revalorisation de la valeur du point conventionnel à effet rétroactif au 1er juillet 2022 ne bénéficiera pas au personnel dont le coefficient et donc la rémunération reste à la valeur du SMIC, a répondu favorablement pour le versement d’une prime de 150 euros brute exceptionnelle pour les personnels concernés par ces coefficients au prorata de leur temps de travail contractuel. Cette prime permettra de «\xa0compenser\xa0» la régularisation de la différence sur la valeur du point du 1er juillet au 31 décembre 2022 dont ils ne pourront bénéficier, et sera versée en une fois, en même temps que la régularisation de la valeur du point faite pour les autres membres du personnel sur la paie de janvier 2023.\n\nLes bénéficiaires de la mesure sont tous les professionnels qu’ils soient à temps complet ou à temps partiel, en contrat à durée indéterminée ou en contrat à durée déterminée, qui, au 1er juillet 2022, après application de la valeur du point résultant de la recommandation patronale FEHAP du 23 novembre 2022, sont concernés par l’application de l’article 08-02 de la CCN51 relatif au salaire minimum conventionnel.\n\nCette prime est exclue de l’assiette de calcul de toutes les primes et indemnités prévues par la Convention Collective nationale du 31 octobre 1951.',
 '2-3\xa0: Récupération des heures de fériés et fixation du jour de solidarité pour 2023\xa0:': '2-3\xa0: Récupération des heures de fériés et fixation du jour de solidarité pour 2023\xa0:\n\nLa recommandation patronale du 4 septembre 2012\xa0de la CCN51 avait créé 2 catégories de personnel concernant l’avantage du férié récupéré\xa0: le personnel présent au 1er décembre 2011 ayant pu continuer à bénéficier des anciennes dispositions de la convention du fait d’avantages individuels acquis, et le personnel arrivé après le 1er décembre 2011 qui a dû se voir attribuer les nouveaux critères prévus dans la recommandation patronale. Cela a engendré un souci d’équité.\n\nA compter du 1er janvier 2023, tous les salariés, sans condition d’ancienneté, récupéreront les heures de fériés qu’elles soient travaillées ou non selon les modalités prévues dans la recommandation patronale du 4 septembre 2012.\n\nAu 1er janvier 2023, le don de la journée de solidarité se fera par le biais de la suppression d’une récupération de jour férié (hormis celle due au titre du 1er mai éventuellement générée).\nSi le salarié apporte la preuve (bulletin de salaire faisant mention, attestation, …) qu’il a déjà effectué sous quelque forme que ce soit la journée solidarité au titre de l’année concernée auprès d’un autre employeur, il n’aura pas à l’effectuer au sein de la structure.\nLe salarié ayant plusieurs employeurs effectue sa journée de solidarité chez chacun d’eux au prorata de sa durée contractuelle de travail, de ce fait si le salarié apporte la preuve (bulletin de salaire faisant mention, attestation, …) qu’il a effectué au prorata sa journée ou son don pour la journée solidarité, il ne l’effectuera qu’au prorata au sein de la structure.\nLa journée de solidarité sera évoquée sur le bulletin de salaire de manière à pouvoir apporter la preuve qu’elle a été effectuée dans la structure.\nCas du salarié qui n’a pas pu obtenir de récupération de férié\xa0(pas de férié tombant sur un repos, suspension de contrat ou congé payé durant un férié)\xa0: celui-ci donnera un RTT s’il est concerné par ce dispositif. S’il n’en a pas, il pourra donner un repos conventionnel (tel qu’un repos compensateur de nuit par exemple), sinon il effectuera 7 heures supplémentaires (ou moins selon son temps contractuel) selon les modalités à convenir avec son supérieur hiérarchique de manière à valider son don pour la journée de solidarité.',
 '2-4 : Revalorisation des heures supplémentaires\xa0:': '2-4 : Revalorisation des heures supplémentaires\xa0:\n\nAfin de récompenser les salariés qui accepteraient de remplacer un collègue absent au «\xa0pied levé\xa0», les parties ont convenu de valoriser les heures supplémentaires à hauteur de 150% au lieu de 125% pour toute demande effectuée le vendredi pour le week-end et le lundi, et 24h avant en semaine.',
 '2-5\xa0: Prime parrainage\xa0:': '2-5\xa0: Prime parrainage\xa0:\n\nLa prime de parrainage accordée en 2022 pour toute aide au recrutement de la part des salariés par présentation d’un candidat n’est pas reconduite pour l’année 2023.\nToutefois, une prime de parrainage de 2\xa0500€ brut, est accordée pour toute aide au recrutement d’un médecin praticien d’HAD (0,80 à 1 ETP) et versée à la fin de la période d’essai du professionnel.',
 'Article 3\xa0: conditions de travail': 'Article 3\xa0: conditions de travail\n\n3-1\xa0: Casiers nominatifs sur chaque antenne\xa0:\n\nDe nouvelles antennes et locaux ont été aménagés en 2022. Pour répondre à la problématique d’accueil de nouveaux collaborateurs et le travail en mobilité sur plusieurs antennes, les parties se sont accordées sur l’agencement de bureaux partagés nécessitant la mise à disposition de casiers nominatifs au sein de chaque antenne.\nLa direction s’engage à réaliser les achats nécessaires pour la mise à disposition de ces casiers nominatifs au sein de chaque antenne dès l’agencement terminé.',
 '3-2\xa0: pause déjeuner du personnel\xa0:': '3-2\xa0: pause déjeuner du personnel\xa0:\n\nLa demande des salariés est de réduire le temps de présence journalier sur leur lieu de travail et de diminuer le temps accordé à la pause repas à 30 mn au lieu d’une heure.\n\nLes parties s’accordent sur une pause de 30 mn à condition que cela n’affecte pas le fonctionnement du service. Les horaires de travail seront ajustés par les responsables en fonction de l’amplitude de la pause repas et devront correspondre aux besoins de l’établissement.',
 '3-3\xa0: utilisation voitures de service\xa0:': '3-3\xa0: utilisation voitures de service\xa0:\n\nLe personnel soignant pourra garder le véhicule de service en cas de travail sur 2 jours consécutifs, par nécessité de service. En contrepartie, le salarié s’engage, par tout moyen, à restituer le véhicule de service en cas d’absence non programmée. Cf modalités dans le règlement intérieur des véhicules de service signé par le personnel avec attestation de remisage.',
 'Article 4 – Information du Comité Social et Economique': 'Article 4 – Information du Comité Social et Economique\nLe CSE sera informé du présent accord lors de réunion du 19 janvier 2023, dans le cadre de sa mission au titre de l’article L2312-8 du code du travail.',
 'Article 5 - Dispositions relatives à l’accord ': 'Article 5 - Dispositions relatives à l’accord \nLe présent accord entre en application après son dépôt sur la plateforme de téléprocédure en application des conditions légales et réglementaires en vigueur, pour une durée indéterminée.\nLe présent accord est également déposé au secrétariat-greffe du Conseil des Prud’hommes de Libourne.\nIl pourra être révisé conformément aux dispositions légales.\nIl fait l’objet des mesures de publicité prévues par les dispositions légales et réglementaires sur les lieux d’affichage habituels.\n\nFait à Libourne, le 19 janvier 2023, \n\n\nSignature de la Direction\xa0:\n\n\nSignatures des membres titulaires du CSE\xa0:'}

In [31]:
mon_dict ={'ACCORD COLLECTIF D’ENTREPRISE RELATIF A L’AUGMENTATION': 'ACCORD COLLECTIF D’ENTREPRISE RELATIF A L’AUGMENTATION',
 'DU CONTINGENT D’HEURES SUPPLEMENTAIRES ': 'DU CONTINGENT D’HEURES SUPPLEMENTAIRES \n\n\nEntre les soussignés :\n\nLa société PIEPLU PAUL, Société à Responsabilité Limitée, dont le siège social est situé à SANNERVILLE (14940), 20 Rue du 6 Juin, immatriculée au RCS de CAEN sous le N°483\xa0527\xa0644, représentée aux présentes par Monsieur , Gérant\n\nD’une part\nEt,\nLes membres du personnel statuant à la majorité des deux tiers,\n\nD’autre part\n\n\nIl a été arrêté et convenu ce qui suit\xa0:',
 'PREAMBULE ET OBJET DE L’ACCORD': 'PREAMBULE ET OBJET DE L’ACCORD\n\n\nLes impératifs de l’activité de notre société, qui relève de la Convention Collective Nationale des Ouvriers du Bâtiment (IDCC 1596), des ETAM du Bâtiment (IDCC 2609) et des Cadres du Bâtiment (IDCC 2420), oblige la société à recourir à l’accomplissement par ses salariés d’heures supplémentaires de manière récurrente. \n\nA ce jour, le contingent annuel d’heures supplémentaires prévu par les conventions collectives est fixé à 180 heures par an et par salarié, ce qui se révèle réellement inadapté aux besoins et aux impératifs de notre charge de travail. \n\nCompte tenu des difficultés de recrutement dans la profession et d’une volonté d’assurer la réalisation des chantiers dans les délais impartis, tout en assurant la protection des droits des salariés, les parties ont convenu d’adopter, par le présent accord, un contingent annuel d’heures supplémentaires supérieur à celui prévu par les Conventions Collectives Nationales du Bâtiment.\n\nAu-delà de l’augmentation du contingent annuel d’heures supplémentaires, le présent accord a pour objet de fixer les contreparties prévues pour les heures supplémentaires effectuées dans le cadre dudit contingent, et de fixer les modalités de dépassement éventuel du contingent d’heures supplémentaires et de prise des contreparties en repos des heures supplémentaires effectuées au-delà dudit contingent.',
 'ARTICLE 1 — CHAMP D’APPLICATION': 'ARTICLE 1 — CHAMP D’APPLICATION',
 'ARTICLE 1.1\xa0– Salariés concernés': 'ARTICLE 1.1\xa0– Salariés concernés\n\nLes dispositions du présent accord s’appliquent à l’ensemble du personnel de la société (Ouvriers, ETAM et Cadres)° employé à temps complet en contrat à durée déterminée ou indéterminée.',
 'ARTICLE 2 — ENTREE EN VIGUEUR ET DUREE DE L’ACCORD': 'ARTICLE 2 — ENTREE EN VIGUEUR ET DUREE DE L’ACCORD\n\n\nLe présent accord entrera en vigueur rétroactivement à compter du 01 Janvier 2022.\nIl est conclu pour une durée indéterminée. \n\nIl pourra être dénoncé dans les conditions prévues à l’article 5 du présent accord.',
 'ARTICLE 3 — CONTINGENT ANNUEL D’HEURES SUPPLEMENTAIRES': 'ARTICLE 3 — CONTINGENT ANNUEL D’HEURES SUPPLEMENTAIRES',
 'Article 3-1 – Fixation du contingent d’heures supplémentaires ': 'Article 3-1 – Fixation du contingent d’heures supplémentaires \n\nA compter du 01 Janvier 2022, le contingent annuel d’heures supplémentaires est porté à 300 heures par an et par salarié.\n\nLa période de référence pour calculer le contingent d’heures supplémentaires est\xa0l’année civile, soit du 1er janvier de l’année N au 31 décembre de l’année N. \n\nLes heures prises en compte pour le calcul du contingent annuel d’heures supplémentaires sont celles accomplies au-delà de la durée légale applicable au sein de la société et donnant lieu à une majoration de salaire. \nS’imputent donc sur ledit contingent, les heures supplémentaires effectuées et payés par les salariés. \nSont par conséquent exclues de ce contingent d’heures supplémentaires, les heures supplémentaires non rémunérées et compensées intégralement par un repos.\nCe contingent s’applique sans prorata temporis pour les salariés embauchés en cours d’année.\nL’utilisation de ce contingent d’heures supplémentaires se fera dans le respect des règles relatives aux temps de repos minimum et au temps de travail effectif maximum.',
 'Article 3-2 – Contreparties des heures supplémentaires effectuées à l’intérieur du contingent ': 'Article 3-2 – Contreparties des heures supplémentaires effectuées à l’intérieur du contingent \n\n\nConformément aux dispositions légales et conventionnelles actuellement en vigueur, les heures supplémentaires effectuées au-delà de la durée hebdomadaire de travail de 35 heures par semaine, ouvrent droit à une majoration de\xa0:\n25% du salaire horaire effectif pour les 8 premières heures,\nEt 50% du salaire horaire effectif au-delà de la 8ème heure.',
 'ARTICLE 4 — REVISION DE L’ACCORD': 'ARTICLE 4 — REVISION DE L’ACCORD\n\n\nLe présent accord pourra être révisé, à compter d’un délai d’application de 6 mois, dans les conditions prévues par la loi.\n\nLes conditions de validité des avenants de révision sont identiques à celles des accords initiaux.',
 'ARTICLE 5 — DENONCIATION DE L’ACCORD': 'ARTICLE 5 — DENONCIATION DE L’ACCORD\n\n\n\nLe présent accord, conclu sans limitation de durée, pourra être dénoncé à tout moment par l’une ou l’autre des parties signataires sous réserve de respecter un préavis de 3 mois.\nCette dénonciation devra être notifiée à l’ensemble des autres signataires par lettre recommandée avec avis de réception.\nDans ce cas, la Direction et le personnel se réuniront pendant la durée du préavis pour discuter les possibilités d’un nouvel accord.',
 'ARTICLE 6 — PUBLICITE ET DEPOT DE L’ACCORD': 'ARTICLE 6 — PUBLICITE ET DEPOT DE L’ACCORD\n\n\n\nLe présent accord sera déposé sur la plateforme en ligne TéléAccords. \nEn outre, conformément aux dispositions légales en vigueur, le présent accord sera rendu public dans son intégralité et accessible dans la base de données nationale prévue à cet effet\xa0: https://www.legifrance.gouv.fr/. A cet effet, une version de l’accord déposé en format Word dans laquelle toute mention de noms, prénoms de personnes physiques y compris les paraphes et les signatures sont supprimées.\n\nUn exemplaire de l’accord sera également remis au greffe du conseil de prud’hommes de CAEN.\n\nFait à Sannerville \nEn 3 exemplaires originaux\nLe 28/03/2022\n\nLE PRESIDENT\nMonsieur \n\n\n\nLES SALARIES\nMadame \t\t\t\t\tMonsieur \n\n\n\nMonsieur \t\t\t\t\tMonsieur \n\n\n\nMonsieur \t\t\t\t\tMonsieur \n\n\n\nMonsieur \t\t\t\t\tMonsieur \n\n\n\nMonsieur \t\t\t\t\tMonsieur \n\n\n\n\nMadame \t\t\t\t\tMadame \n\n\n\n\nMadame \t\t\t\t\tMonsieur \n\n\n\nMonsieur'}

In [32]:
mon_dict ={'Préambule ': 'Préambule \n\nLe présent accord ne constitue pas une approbation explicite ou implicite par les organisations syndicales signataires du nouveau schéma industriel choisi par La Poste ni de ses conséquences en terme de niveau d’emploi.',
 'Article 1 – Champ d’application': "Article 1 – Champ d’application\n\nLe présent accord mettant en place une organisation du temps de travail sur plusieurs semaines est  applicable à l'ensemble du personnel, fonctionnaires, salariés et ACO de droit public, affectés à GUEUGNON du site de CHAROLLES\n\nIl est convenu que les régimes de travail mis en place dans le cadre du présent accord et prévus pour le personnel susvisé, se substituent aux anciens régimes de travail résultant d’usage(s) ou d’accord(s) jusqu’alors en vigueur dans l’établissement courrier de GUEUGNON, site de CHAROLLES\n\nL’organisation du temps de travail instituée par le présent accord est strictement liée au site de CHAROLLES pris en tant qu’entité géographique. Elle n’est applicable pour les activités susvisées que si celles-ci sont exercées sur le site de CHAROLLES.",
 'Article 2 – Durée du travail': 'Article 2 – Durée du travail\n\nLa durée du travail applicable aux personnels visés à l’article 1, conformément aux articles L.3122-1 et suivants du Code du travail et à l’accord cadre du 17 février 1999 est de 35 heures hebdomadaire en moyenne sur la période définie à l’article 3 du présent accord.',
 'Article 3 – Aménagement du temps de travail': 'Article 3 – Aménagement du temps de travail\n\n«La durée de travail définie à l’article 2 du présent accord est répartie dans le cadre d’une période de référence.\nSur la durée totale de la période, les agents travaillent en moyenne 35 heures sur chaque période. \n\nEquipe Cabine \n\nPériode du 19/06/2018 au 15/06/2020, les agents travaillent en moyenne 35 heures par période de 12 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 3 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 4 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 5 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 6 avec une durée hebdomadaire de travail (DHT) de 17 heures 38 mn, avec  3  jours de repos\xa0: le lundi, le mardi, le mercredi\nSemaine 7 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 8 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 9 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine10 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 11 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn \nSemaine 12 avec une durée hebdomadaire de travail (DHT) de 20 heures 33 mn, avec  3  jours de repos\xa0: le jeudi, le vendredi et le samedi\n\n\nEquipe ROP\n\nPériode du 19/06/2018 au 15/06/2020, les agents travaillent en moyenne 35 heures par période de 2 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  36 heures 10 mn avec 1 jour de repos le mercredi\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  33 heures 50 mn avec 1 jour de repos le samedi\n\nEquipe distribution 2\n\nPremière période du 19/06/2018 au XX/XX/2019, les agents travaillent en moyenne 35 heures par période de 12 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 3 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 4 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 5 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 6 avec une durée hebdomadaire de travail (DHT) de 17 heures 18  mn, avec  3  jours de repos\xa0: le lundi, le mardi et le mercredi\nSemaine 7 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 8 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 9 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine10 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 11 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn \nSemaine 12 avec une durée hebdomadaire de travail (DHT) de 20 heures 51 mn, avec  3  jours de repos\xa0: le jeudi, le vendredi et le samedi\n\nCependant en cas d’évolution des activités dûment constatée dans le SI (TRTP Trafic de référence Tous Produits) supérieure ou égale à 6 % en moyenne\xa0lors d’une commission de suivi; \nPar rapport au TRTP constaté à la mise en œuvre de l’organisation\nSur une période de 12\xa0 mois glissants et sous réserve d’un délai de prévenance d’un mois,\nLa période de référence sera modifiée et portée à 18 semaines.\n\nDeuxième période du XX/XX/2019 au 15/06/2020, les agents travaillent en moyenne 35 par période de 18 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 3 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn \nSemaine 4 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 5 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 6 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 7 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 8 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 9 avec une durée hebdomadaire de travail (DHT) de  16 heures 51 mn, avec  3  jours de repos\xa0: le lundi, le mardi et le mercredi\nSemaine 10  avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 11 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 12 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 13 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 14 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 15 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 16 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn \nSemaine 17 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 18 avec une durée hebdomadaire de travail (DHT) de 20 heures 12 mn, avec  3  jours de repos\xa0: le jeudi, le vendredi et le samedi\n\n\nEquipes distribution 3, 4 et 5\n\nPremière période du 19/06/2018 au XX/XX/2019, les agents travaillent en moyenne 35 heures par période de 12 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 3 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 4 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 5 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 6 avec une durée hebdomadaire de travail (DHT) de 18 heures 00  mn, avec  3  jours de repos\xa0: le lundi, le mardi et le mercredi\nSemaine 7 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 8 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 9 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine10 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn\nSemaine 11 avec une durée hebdomadaire de travail (DHT) de  38heures 11 mn \nSemaine 12 avec une durée hebdomadaire de travail (DHT) de 20 heures 09 mn, avec  3  jours de repos\xa0: le jeudi, le vendredi et le samedi\n\nCependant en cas d’évolution des activités dûment constatée dans le SI (TRTP Trafic de référence Tous Produits) supérieure ou égale à 6 % en moyenne\xa0lors d’une commission de suivi; \nPar rapport au TRTP constaté à la mise en œuvre de l’organisation\nSur une période de 12\xa0 mois glissants et sous réserve d’un délai de prévenance d’un mois,\nLa période de référence sera modifiée et portée à 18 semaines.\n \n\n\nDeuxième période du XX/XX/2019 au 15/06/2020, les agents travaillent en moyenne 35 par période de 18 semaines, selon les modalités suivantes\xa0:\n\nSemaine 1 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 2 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 3 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn \nSemaine 4 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 5 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 6 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 7 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 8 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 9 avec une durée hebdomadaire de travail (DHT) de  17 heures 17 mn, avec  3  jours de repos\xa0: le lundi, le mardi et le mercredi\nSemaine 10  avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 11 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 12 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 13 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 14 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 15 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 16 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn \nSemaine 17 avec une durée hebdomadaire de travail (DHT) de  37 heures 04 mn\nSemaine 18 avec une durée hebdomadaire de travail (DHT) de 19 heures 36 mn, avec  3  jours de repos\xa0: le jeudi, le vendredi et le samedi\n\n\n\n\nLa répartition du travail au sein de chaque période de référence ainsi que les horaires collectifs de travail afférents à ces régimes de travail sont communiqués aux agents par affichage dans l’établissement. \nLa durée de travail, les dates et jours de repos, ainsi que les horaires de travail peuvent être modifiés par l’employeur, en cas de nécessité liée au service ou de contraintes de production, sous réserve de respecter un délai de prévenance de 30 jours.',
 'Article 4 – Heures supplémentaires': 'Article 4 – Heures supplémentaires',
 '4.1 Définition': '4.1 Définition\n\nConstituent des heures supplémentaires, les heures effectuées au-delà de la moyenne de 35 heures calculée sur chaque période prévue à l’article 3 du présent accord.\n\n\n4.2 Les heures supplémentaires accomplies au-delà de la moyenne de 35h00 calculées sur la période de référence.\n\nLe paiement de ces heures et des majorations y afférentes sera\xa0au choix de l’agent : \n\nSoit remplacé par un  repos compensateur équivalent, auquel cas les heures supplémentaires ne s’imputent pas sur le contingent d’heures supplémentaires  conformément aux dispositions légales ou réglementaires applicables selon le statut de chaque agent concerné.\n\nSoit  effectué conformément aux dispositions légales ou réglementaires applicables selon le statut de chaque agent concerné, à savoir paiement en salaire majoré et imputation sur le contingent d’heures supplémentaires.',
 'Article 5 – Rémunération': "Article 5 – Rémunération\n\nAfin d'éviter toute variation de rémunération, le salaire de base sera indépendant de l'horaire réellement effectué dans la semaine : la rémunération sera lissée sur le mois.\n\nLes agents seront rémunérés sur la base de 35 heures par semaine soit sur 151,67 heures par mois. \n\nLes éventuelles absences non rémunérées et les heures supplémentaires sont comptabilisées à l’issue de la période de référence.",
 'Article 6 – Embauche ou rupture de contrat de travail au cours de la période de référence': 'Article 6 – Embauche ou rupture de contrat de travail au cours de la période de référence\n\nSauf clause contraire prévue au contrat de travail, les agents embauchés en cours de période de référence suivent les horaires en vigueur dans l’entreprise.\nA la fin de la période durant laquelle l’agent a été embauché, il est procédé à une régularisation sur la base d’un temps réel de travail au cours de la période de présence par rapport à 35 heures hebdomadaires.\n\nEn cas de rupture du contrat de travail, la rémunération sera régularisée sur la base des heures effectivement travaillées\xa0:\n - la rémunération ne correspondant pas à du temps de travail effectif sera prélevée sur le dernier bulletin de salaire\xa0conformément aux dispositions légales et réglementaires.\n - les heures excédentaires par rapport à 35 heures seront payées au salarié avec les bonifications et les majorations applicables aux heures supplémentaires.',
 ' Article 7 – Salariés à temps partiel': 'Article 7 – Salariés à temps partiel\n\nLes salariés à temps partiel affectés aux services de CHAROLLES sont soumis à l’organisation du temps de travail institués par le présent accord.\n\nLa répartition de la durée du travail sur la période définie à l’article 3 du présent accord ainsi que les horaires journaliers de travail sont communiqués à ces salariés, individuellement. Ils peuvent faire l’objet d’une modification en raison des contraintes de production, sous réserve de respecter un délai de prévenance de 30 jours calendaires. \n\nL’application de cette disposition est réalisée sans préjudice des dispositions contractuelles figurant dans les contrats de travail des salariés concernés à la date d’entrée en vigueur du présent accord.',
 'Article 8 – Durée de l’accord, révision, dénonciation': 'Article 8 – Durée de l’accord, révision, dénonciation\n\nLe présent accord entrera en vigueur à compter du 20/03/2018 sous réserve de l’absence d’opposition majoritaire, et cessera de plein droit de produire tout effet à son terme fixé au 15/06/2020.\n\nL’accord signé sera notifié par lettre recommandée avec accusé de réception aux organisations syndicales représentatives non signataires et signataires.\n\nChaque partie signataire ou adhérente peut demander la révision de tout ou partie du présent accord, selon les modalités prévues par l’accord national du 21 juin 2004 sur les principes et méthodes du dialogue social à La Poste.\n\nEn cas de modification des dispositions légales ou conventionnelles relatives au temps de travail, les parties signataires se réuniront, à l’initiative de la partie la plus diligente, dans un délai de 3 mois à compter de la date d’entrée en vigueur des nouvelles dispositions légales ou conventionnelles, afin d’examiner les aménagements à apporter au présent accord.'
 }

In [43]:
#print(get_valid_chunks_filtered(mon_dict, seuil_sim=0.85))

In [86]:
print(get_valid_title_filtered(mon_dict, seuil_sim=0.85))

['– Durée du travail', '– Aménagement du temps de travail', '– Heures supplémentaires', 'Définition', '– Rémunération', '– Embauche ou rupture de contrat de travail au cours de la période de référence', '– Salariés à temps partiel']


# Pour HS

In [34]:
sommaire_hs = pd.read_parquet("data/echantillon_1000_hs_accords_TOC.parquet")
df_hs = pd.read_parquet("data/echantillon_1000_hs_accords.parquet")
df_hs = df_hs.set_index("numdossier_new")
df_hs = df_hs.merge(sommaire_hs,how="inner",left_index=True,right_index=True)
df_hs = df_hs.rename(columns={"extracted_summary":"summary"})

In [38]:
df_hs["section_dict"] = df_hs.apply(
    lambda row: split_text_with_titles(row["accorddocx"], row["summary"]),
    axis=1
)
df_hs["section_dict"][0]

/tmp/ipykernel_1833/3173179327.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_hs["section_dict"][0]


{'Préambule :  ': 'Préambule :  \n\nConformément aux dispositions du code du travail, la Direction a invité le CSE, en l’absence d’organisations syndicales représentatives dans la structure  à participer à une négociation collective.\n\nAux termes des réunions en date des 10/10/2022, 16/11/2022 et 12/12/2022 ayant permis de rapprocher les points de vue de chacun, les parties ont abouti à la conclusion du présent accord.',
 'Article 1 – Champ d’application et bénéficiaires\xa0:  ': 'Article 1 – Champ d’application et bénéficiaires\xa0:  \n\nLe présent accord concerne l’ensemble des établissements de l’HADVR.\n\nIl concerne tous les salariés quel que soit leur contrat (CDD ou CDI), quelle que soit leur durée de travail et quel que soit leur métier.\nPar ailleurs, pour répondre aux aspirations des salariés d’une part et aux contraintes inhérentes d’une HAD, les parties se sont accordées pour poursuivre les négociations tout au long de l’année 2023 en vue de la conclusion éventuelle d’un a

In [39]:
df_hs["lda_documents"] = df_hs["section_dict"].apply(get_valid_chunks_filtered)

In [40]:
# Nettoyage NLP + lemming
all_chunks_hs = [chunk for doc_chunks in df_hs["lda_documents"] for chunk in doc_chunks]
processed_texts_hs = [preprocess_text(doc) for doc in all_chunks_hs]

In [41]:
df_processed_texts_hs = pd.DataFrame({"processed_texts_hs": processed_texts_hs})
df_processed_texts_hs.to_parquet(f"{OUTPUT_DIR}/processed_texts_hs.parquet", index=False)

In [42]:
df_hs[["section_dict", "lda_documents"]].to_csv(f"{OUTPUT_DIR}/trie_titres.csv", index=False)

# Pour les données de santé

In [ ]:
df_sante= pd.read_parquet("data/complementaire_sante_580.parquet")


In [ ]:
df_sante["section_dict"] = df_sante.apply(
    lambda row: split_text_with_titles(row["accorddocx"], row["extracted_summary"]),
    axis=1
)


In [ ]:
df_sante["lda_documents"] = df_sante["section_dict"].apply(get_valid_chunks_filtered)

In [ ]:
# Nettoyage NLP + lemming
all_chunks_sante = [chunk for doc_chunks in df_sante["lda_documents"] for chunk in doc_chunks]
processed_texts_sante = [preprocess_text(doc) for doc in all_chunks_sante]

In [ ]:
df_processed_texts_sante = pd.DataFrame({"processed_texts_sante": processed_texts_sante})
df_processed_texts_sante.to_parquet(f"{OUTPUT_DIR}/processed_texts_sante.parquet", index=False)